# Phase 34 — v1.3 Model Identity + Untimed Smoke Gate
## Reviewer Notebook

**Classification:** `READ_ONLY_PHASE34_V1_3_MODEL_IDENTITY_AND_UNTIMED_SMOKE_GATE_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase34_efficiency_profiling_v1_3`

All four canonical components were constructed, strict-loaded and parameter-verified, and ten
untimed smoke forwards were executed (4 component + 2 SKIP path + 4 EXECUTE path). **No latency,
throughput, memory or profiler measurement was performed.**


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


print("Phase34 v1.3 model-identity + untimed smoke-gate review")
print("READ-ONLY. No checkpoint load, no model construction, no forward, no timing,")
print("no memory measurement, no protected data, no TEST, no training.")


Phase34 v1.3 model-identity + untimed smoke-gate review
READ-ONLY. No checkpoint load, no model construction, no forward, no timing,
no memory measurement, no protected data, no TEST, no training.


## 1. Runtime source preflight (additive filename)

In [2]:
rp = jload(AUD / "phase34_execution_source_preflight_runtime_v1_3.json")
print("additive filename:", rp["additive_filename"])
print("authority all match:", rp["authority_all_match"])
print("frozen sources all match:", rp["frozen_sources_all_match"])
print("frozen preflight unchanged:", rp["frozen_preflight_artifact"]["unchanged"])
print()
for f in rp["frozen_source_verification"]:
    print(f"  {'OK ' if f['match'] else 'BAD'} {f['key']:26s} {f['observed_sha256'][:16]}...")


additive filename: True
authority all match: True
frozen sources all match: True
frozen preflight unchanged: True

  OK  common                     ca74fe88a8314b60...
  OK  worker                     295c64d94358aed7...
  OK  harness                    3244c2f95fbd1381...
  OK  preflight                  494925ecbc9a13eb...
  OK  import_probe               3b55cf95980893bb...
  OK  model_G                    a04ff3e61b4d900f...
  OK  model_R_psi                4fe0abde3ec3a183...
  OK  model_B_phi_authority      fc7c1bdf263ae455...
  OK  model_S_omega_authority    98cf958a377e2de3...
  OK  domain_task_registry       3f009d994ed24d9c...


## 2. Environment gate

In [3]:
env = jload(AUD / "phase34_smoke_execution_environment_v1_3.json")
print("gate pass:", env["environment_gate_pass"])
for k, v in env["environment_checks"].items():
    print(f"  {k:28s} = {v}  (observed: {env['observed_environment'].get(k)})")


gate pass: True
  gpu_model                    = True  (observed: NVIDIA GeForce RTX 3060 Ti)
  gpu_compute_capability       = True  (observed: 8.6)
  gpu_total_vram_bytes         = True  (observed: 8589279232)
  python_version               = True  (observed: 3.11.9)
  torch_version                = True  (observed: 2.6.0+cu124)
  cuda_runtime                 = True  (observed: 12.4)
  cuda_driver_version          = True  (observed: 596.36)
  cudnn_version                = True  (observed: 90100)


## 3. Model identity per component

In [4]:
sg = jload(AUD / "phase34_model_identity_and_smoke_gate_v1_3.json")
print("checkpoint hashes all match:", sg["all_checkpoint_hashes_match"])
print("execution mode:", sg["execution_mode"],
      "| harness used:", sg["orchestration_harness_used"])
print()
rows = []
for cid, v in sg["per_component"].items():
    rows.append({"component": cid, "constructed": v["constructed"],
                 "missing_keys": len(v["missing_keys"] or []),
                 "unexpected_keys": len(v["unexpected_keys"] or []),
                 "params": v["parameter_count"], "dtype": ",".join(v["parameter_dtypes"] or []),
                 "eval": v["eval"], "out_shape": str(v["output_shape"]),
                 "finite": v["all_finite"]})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("total resident parameters:", sum(v["parameter_count"] or 0
                                        for v in sg["per_component"].values()))


checkpoint hashes all match: True
execution mode: smoke_only | harness used: False

component  constructed  missing_keys  unexpected_keys  params         dtype  eval        out_shape  finite
        G         True             0                0 3523001 torch.float32  True [1, 1, 512, 512]    True
    R_psi         True             0                0 1276401 torch.float32  True [1, 1, 256, 256]    True
    B_phi         True             0                0   36801 torch.float32  True              [1]    True
  S_omega         True             0                0    1217 torch.float32  True              [1]    True

total resident parameters: 4837420


## 4. Smoke forward accounting

In [5]:
print("SMOKE_TASK_ID:", sg["SMOKE_TASK_ID"], "| SMOKE_DOMAIN:", sg["SMOKE_DOMAIN"])
print("individual component smoke forwards:", sg["individual_smoke_forwards"])
print("SKIP controlled-path forwards     :", sg["SKIP_smoke_forwards"])
print("EXECUTE controlled-path forwards  :", sg["EXECUTE_smoke_forwards"])
print("TOTAL smoke model forward calls   :", sg["total_smoke_model_forward_calls"])
print("smoke failures:", sg["smoke_failures"])
print("all smoke outputs finite:", sg["all_smoke_outputs_finite"])
print()
print("CUDA synchronization note:", sg["cuda_synchronization_note"])


SMOKE_TASK_ID: 0 | SMOKE_DOMAIN: ISIC
individual component smoke forwards: 4
SKIP controlled-path forwards     : 2
EXECUTE controlled-path forwards  : 4
TOTAL smoke model forward calls   : 10
smoke failures: []
all smoke outputs finite: True

CUDA synchronization note: the frozen worker surfaces asynchronous CUDA errors via the blocking torch.isfinite(out).all().item() device->host transfer in validate_output; it does NOT create CUDA timing events and produces NO timing statistics


## 5. Controlled-path smoke

In [6]:
for arm in ("SKIP_CONTROLLED_PATH", "EXECUTE_CONTROLLED_PATH"):
    r = sg[arm.replace("_CONTROLLED_PATH", "_smoke")]
    print(arm, "| mode:", r["mode"], "| scope:", r["scope"])
    p = r["path"]
    print("   components:", p["components"], "| forwards:", p["forward_calls"],
          "| route_forced:", p["route_forced"],
          "| b_phi_chose_route:", p["b_phi_output_used_to_choose_route"])
    for cid, o in p["outputs"].items():
        print(f"     {cid:8s} shape={o['output_shape']} dtype={o['output_dtype']} "
              f"finite={o['finite']}")


SKIP_CONTROLLED_PATH | mode: smoke | scope: CONTROLLED_PATH
   components: ['G', 'B_phi'] | forwards: 2 | route_forced: True | b_phi_chose_route: False
     G        shape=[1, 1, 512, 512] dtype=torch.float32 finite=True
     B_phi    shape=[1] dtype=torch.float32 finite=True
EXECUTE_CONTROLLED_PATH | mode: smoke | scope: CONTROLLED_PATH
   components: ['G', 'B_phi', 'R_psi', 'S_omega'] | forwards: 4 | route_forced: True | b_phi_chose_route: False
     G        shape=[1, 1, 512, 512] dtype=torch.float32 finite=True
     B_phi    shape=[1] dtype=torch.float32 finite=True
     R_psi    shape=[1, 1, 256, 256] dtype=torch.float32 finite=True
     S_omega  shape=[1] dtype=torch.float32 finite=True


## 6. Smoke execution provenance

In [7]:
pv = jload(AUD / "phase34_smoke_execution_provenance_v1_3.json")
print("runtime preflight SHA:", pv["runtime_preflight_sha256"][:16], "...")
print("v1.3 source freeze SHA:", pv["v1_3_source_freeze_sha256"][:16], "...")
print("SMOKE_TASK_ID:", pv["SMOKE_TASK_ID"], "| generator:", pv["synthetic_generator"])
print("smoke order:", pv["smoke_order"])
print("forward accounting:", pv["forward_call_accounting"])
print()
print("protected-data counters:")
for k, v in pv["protected_data_counters"].items():
    print(f"  {k:30s} = {v}")
print()
print("precision contexts:")
for k, v in pv["precision_contexts"].items():
    print(f"  {k:8s} {v}")


runtime preflight SHA: c158e57db219519d ...
v1.3 source freeze SHA: c7ee684c24d3ddaf ...
SMOKE_TASK_ID: 0 | generator: PYTORCH_CUDA_GENERATOR_SEED_42
smoke order: ['G_COMPONENT', 'R_PSI_COMPONENT', 'B_PHI_COMPONENT', 'S_OMEGA_COMPONENT', 'SKIP_CONTROLLED_PATH', 'EXECUTE_CONTROLLED_PATH']
forward accounting: {'individual': 4, 'controlled_path': 6, 'total': 10}

protected-data counters:
  raw_image_reads                = 0
  raw_gt_reads                   = 0
  router_holdout_raw_reads       = 0
  test_reads                     = 0
  training_data_image_reads      = 0
  external_image_reads           = 0
  training_steps                 = 0

precision contexts:
  G        eval + float32 + autocast(float16) + torch.no_grad() + inference_mode false
  R_psi    eval + float32 + autocast disabled + torch.inference_mode()
  B_phi    eval + float32 + autocast disabled + torch.inference_mode()
  S_omega  eval + float32 + autocast disabled + torch.inference_mode()


## 7. Zero-measurement verification

In [8]:
print("latency measurements        :", sg["latency_measurements"])
print("throughput measurements     :", sg["throughput_measurements"])
print("memory measurements         :", sg["memory_measurements"])
print("MAC/FLOP profiler executions:", sg["mac_flop_profiler_executions"])
print("latency warmups             : 0")
print()
print("no latency table  :", not (ROOT / "reports/tables/phase34_batch1_latency_results_v1_3.csv").exists())
print("no throughput table:", not (ROOT / "reports/tables/phase34_batch8_throughput_results_v1_3.csv").exists())
print("no memory table    :", not (ROOT / "reports/tables/phase34_gpu_memory_results_v1_3.csv").exists())


latency measurements        : 0
throughput measurements     : 0
memory measurements         : 0
MAC/FLOP profiler executions: 0
latency warmups             : 0

no latency table  : True
no throughput table: True
no memory table    : True


## 8. Smoke final check and terminal

In [9]:
fc = jload(AUD / "phase34_smoke_final_check_v1_3.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
term = jload(AUD / "phase34_smoke_execution_terminal_v1_3.json")
print()
print("terminal:")
print(" ", term["terminal"])


final check: 41 checks, 0 failed

terminal:
  PHASE34_V1_3_SMOKE_GATE_COMPLETE__ALL_FOUR_CANONICAL_COMPONENTS_STRICT_LOADED_PARAMETER_VERIFIED_AND_UNTIMED_FORWARDS_PASS__CONTROLLED_SKIP_AND_EXECUTE_SMOKES_PASS__TOTAL_SMOKE_FORWARD_CALLS_10__ZERO_LATENCY_THROUGHPUT_MEMORY_OR_PROFILER_MEASUREMENTS__FIRST_MEASURED_PROFILING_EXECUTION_ELIGIBLE_NOT_STARTED


---
## Summary

| item | value |
|---|---|
| G | 3,523,001 params · strict load clean · output `[1,1,512,512]` finite |
| R_psi | 1,276,401 params · strict load clean · output `[1,1,256,256]` finite |
| B_phi | 36,801 params · strict load clean · output `[1]` finite |
| S_omega | 1,217 params · strict load clean · output `[1]` finite |
| total resident parameters | 4,837,420 |
| smoke forwards | 4 component + 2 SKIP + 4 EXECUTE = **10** |
| latency / throughput / memory / profiler | 0 · 0 · 0 · 0 |
| protected data · TEST · training | 0 · 0 · 0 |
| first measured profiling execution | NOT started · Phase35 NOT started |
